# 01 · Del origen a RAW: Extraccion Funcional (V2)

VERSIÓN MEJORADA: Usa rutas absolutas para guardar datos.

Este notebook extrae datos de SQL Server (WideWorldImporters) y los guarda en:
- Archivo CSV en `C:/.../ pipeline_ingesta/datos/bronze/`
- Tabla `raw.invoice_lines` en SQL Server destino

**Salida:** 
- Archivos en `datos/bronze/raw_invoice_lines_LOTE_*.csv`
- Tabla `raw.invoice_lines` en `WideWorldImportersDW2`

In [ ]:
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, BD_ORIGEN, BD_DESTINO
from datetime import datetime
from pathlib import Path

# Obtener engines
engine_origen, engine_destino = obtener_engines()

# Rutas de salida - USAR RUTAS ABSOLUTAS
NOTEBOOK_DIR = Path(__file__).resolve().parent if '__file__' in dir() else Path.cwd()
# Si estamos en Jupyter, usar directorio del notebook
if 'get_ipython' in dir():
    from IPython import get_ipython
    notebook_path = get_ipython().kernel.comm_manager.kernel.shell.user_ns.get('__file__', None)
    if notebook_path:
        NOTEBOOK_DIR = Path(notebook_path).parent
    else:
        # Asumir que estamos en pipeline_ingesta
        NOTEBOOK_DIR = Path.cwd()

DATOS_DIR = NOTEBOOK_DIR / 'datos'
BRONZE_DIR = DATOS_DIR / 'bronze'
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

# ID único de lote
LOTE_ID = f"LOTE_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f'Lote: {LOTE_ID}')
print(f'Origen: {BD_ORIGEN}')
print(f'Destino: {BD_DESTINO}')
print(f'Directorio de salida (ABSOLUTO): {BRONZE_DIR}')

## Paso 1: Crear esquema RAW en destino

In [ ]:
from sqlalchemy import text

sql_crear_raw = """
IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'raw')
    EXEC('CREATE SCHEMA raw');

IF OBJECT_ID('raw.invoice_lines', 'U') IS NULL
BEGIN
    CREATE TABLE raw.invoice_lines (
        id_raw                INT PRIMARY KEY IDENTITY(1,1),
        lote_id               NVARCHAR(50) NOT NULL,
        tabla_origen          NVARCHAR(100) NOT NULL,
        numero_fila           INT NOT NULL,
        payload               NVARCHAR(MAX) NOT NULL,
        hash_sha256           NVARCHAR(64) NOT NULL UNIQUE,
        timestamp_extraccion  DATETIME DEFAULT GETDATE(),
        INDEX IX_lote (lote_id),
        INDEX IX_hash (hash_sha256)
    );
END;
"""

try:
    with engine_destino.begin() as conn:
        conn.execute(text(sql_crear_raw))
    print('[OK] Esquema RAW y tabla creada')
except Exception as e:
    print(f'[ERROR] {e}')

## Paso 2: Extraer datos de origen

In [ ]:
import pandas as pd
import json
import hashlib

# Extraer invoice lines del origen
consulta = """
SELECT TOP 100 
    InvoiceLineID, InvoiceID, StockItemID, 
    Quantity, UnitPrice, TaxRate, TaxAmount, 
    ExtendedPrice, LastEditedWhen
FROM Sales.InvoiceLines
ORDER BY InvoiceLineID
"""

try:
    df_origen = pd.read_sql_query(consulta, engine_origen)
    print(f'[OK] Se extrajeron {len(df_origen)} registros de Sales.InvoiceLines')
    print(f'Columnas: {list(df_origen.columns)}')
    print(f'\nPrimeros 3 registros:')
    display(df_origen.head(3))
except Exception as e:
    print(f'[ERROR] No se pudo leer origen: {e}')
    df_origen = None

## Paso 3: Procesar datos en formato RAW

In [ ]:
if df_origen is not None and len(df_origen) > 0:
    raw_records = []
    
    for idx, row in df_origen.iterrows():
        # Convertir cada fila a diccionario
        record_dict = row.to_dict()
        # Convertir timestamps a string
        for key, val in record_dict.items():
            if pd.isna(val):
                record_dict[key] = None
            elif hasattr(val, 'isoformat'):
                record_dict[key] = val.isoformat()
            else:
                record_dict[key] = str(val)
        
        # Crear payload JSON ordenado
        payload_json = json.dumps(record_dict, sort_keys=True)
        
        # Calcular hash
        hash_sha256 = hashlib.sha256(payload_json.encode('utf-8')).hexdigest()
        
        raw_records.append({
            'lote_id': LOTE_ID,
            'tabla_origen': 'Sales.InvoiceLines',
            'numero_fila': idx + 1,
            'payload': payload_json,
            'hash_sha256': hash_sha256
        })
    
    df_raw = pd.DataFrame(raw_records)
    print(f'[OK] Se procesaron {len(df_raw)} registros en formato RAW')
    print(f'\nEstructura:')
    print(f'- Lote ID: {LOTE_ID}')
    print(f'- Registros: {len(df_raw)}')
    print(f'\nPrimera fila:')
    display(df_raw.head(1))

## Paso 4: Guardar en archivo CSV (RUTA ABSOLUTA)

In [ ]:
if df_raw is not None and len(df_raw) > 0:
    # Archivo CSV con timestamp - RUTA ABSOLUTA
    archivo_csv = BRONZE_DIR / f'raw_invoice_lines_{LOTE_ID}.csv'
    
    try:
        df_raw.to_csv(str(archivo_csv), index=False, encoding='utf-8')
        print(f'[OK] Datos RAW guardados en: {archivo_csv}')
        print(f'Tamaño: {archivo_csv.stat().st_size:,} bytes')
        print(f'Verificación: {archivo_csv.exists()}')
    except Exception as e:
        print(f'[ERROR] {e}')

## Paso 5: Insertar en tabla RAW de destino

In [ ]:
if df_raw is not None and len(df_raw) > 0:
    try:
        df_raw.to_sql(
            'invoice_lines',
            engine_destino,
            schema='raw',
            if_exists='append',
            index=False,
            chunksize=100
        )
        print(f'[OK] {len(df_raw)} registros insertados en raw.invoice_lines')
    except Exception as e:
        print(f'[ERROR] No se pudo insertar en destino: {e}')

## Paso 6: Verificación

In [ ]:
# Verificar archivos en disco
print('[OK] Archivos generados en BRONZE:')
print(f'Directorio: {BRONZE_DIR}')
for archivo in BRONZE_DIR.glob('*.csv'):
    tamano = archivo.stat().st_size
    print(f'  - {archivo.name} ({tamano:,} bytes)')

# Estadísticas en destino
from sqlalchemy import text

try:
    sql_stats = f"""
    SELECT COUNT(*) as total FROM raw.invoice_lines WHERE lote_id = '{LOTE_ID}'
    """
    
    with engine_destino.connect() as conn:
        result = conn.execute(text(sql_stats)).scalar()
        
    print(f'\n[OK] En BD destino:')
    print(f'  Registros en raw.invoice_lines: {result:,}')
except Exception as e:
    print(f'[ERROR] {e}')